# Macro-Aviation-Tourism Demand Modelling for Poland
## Analysis Pipeline — From EDA to Forecasting

**Author:** Diego Marrero Ferrera    
**Repository:** [GitHub](https://github.com/DieGodMF4/Macro-Aviation-Tourism-Modeling)

---

### Objective

Model and forecast **inbound tourism demand to Poland** (measured as monthly overnight stays by foreign tourists) using macroeconomic, price competitiveness, and air connectivity variables from origin markets.

### Roadmap

| Phase | Description | Status |
|-------|-------------|--------|
| **1** | Data loading & initial exploration | ⬜ |
| **2** | Descriptive statistics & univariate analysis | ⬜ |
| **3** | Multivariate EDA (cross-country, correlations) | ⬜ |
| **4** | Statistical tests (stationarity, seasonality, Granger causality) | ⬜ |
| **5** | Feature engineering & selection pipeline | ⬜ |
| **6** | Baseline models (SARIMA, Holt-Winters) | ⬜ |
| **7** | ML models (Random Forest, Gradient Boosting) | ⬜ |
| **8** | Deep Learning — LSTM with exogenous variables | ⬜ |
| **9** | Model comparison, evaluation & scenarios | ⬜ |

### Key references

- Song et al. (2010) — Tourism demand modelling framework, relative price variable methodology
- Salamanis et al. (2022) — LSTM vs LSTMX (with exogenous), long-term tourism forecasting
- Wei et al. (2026) — Two-stage feature selection, Time2Vec-enhanced CNN-BiLSTM
- Xie et al. (2021) — LSSVR-GSA with big data, lag-order analysis for exogenous variables

---
## Phase 0 — Environment Setup

In [2]:
# Core
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Statistical tests
from scipy import stats
from statsmodels.tsa.stattools import adfuller, grangercausalitytests, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Forecasting (baselines)
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ML
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Environment ready.")

Environment ready.


In [9]:
# --- PROJECT PATHS ---
DATA_DIR = os.path.join('data', 'raw')
PROCESSED_DIR = os.path.join('data', 'processed')
FIGURES_DIR = os.path.join('figures')

# Create output dirs if they don't exist
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# --- FILE PATHS ---
PATHS = {
    # Target variable
    'overnight_stays': os.path.join(DATA_DIR, 'tourism', 'tour_occ_nights_accommodation_PL_2003-.csv'),
    
    # Economic
    'hicp_monthly': os.path.join(DATA_DIR, 'economic', 'prc_hicp_midx__relevant-countries-2011-.csv'),
    'exchange_rates': os.path.join(DATA_DIR, 'economic', 'ert_bil_eur_m__exchange-rates-eur-pln-usd-2003-.csv'),
    'real_gdp': os.path.join(DATA_DIR, 'economic', 'namq_10_gdp__real-gdp-CLV-2010.csv'),
    'hicp_annual': os.path.join(DATA_DIR, 'economic', 'prc_hicp_aind__specific-inflation.csv'),
    
    # Demographic
    'consumer_confidence': os.path.join(DATA_DIR, 'demographic', 'ei_bsco_m__consumer-conf-indicator.csv'),
    'population': os.path.join(DATA_DIR, 'demographic', 'demo_pjan__population-eur-990-25.csv'),
    
    # Transport
    'avia_seats': os.path.join(DATA_DIR, 'transport', 'avia_tf_aca__seats-flights-pass-PL-2010-.csv'),
    'avia_passengers': os.path.join(DATA_DIR, 'transport', 'avia_paoc__passengers-countries.csv'),
    
    # Tourism (supplementary)
    'tourism_capacity': os.path.join(DATA_DIR, 'tourism', 'tour_cap_nat__tourism-infraestructure.csv'),
}

# --- COUNTRY DEFINITIONS ---
ORIGIN_COUNTRIES = ['DE', 'UK', 'FR', 'NL', 'IT', 'ES', 'SE', 'AT', 'CZ']
SUBSTITUTE_DESTINATIONS = ['CZ', 'HU', 'HR', 'EL']
DESTINATION = 'PL'
ALL_COUNTRIES = list(set(ORIGIN_COUNTRIES + SUBSTITUTE_DESTINATIONS + [DESTINATION]))

# Eurozone countries (exchange rate cancels out in relative price formula)
EUROZONE = ['DE', 'FR', 'NL', 'IT', 'ES', 'AT', 'EL', 'HR']
NON_EUROZONE = ['UK', 'SE', 'CZ', 'HU', 'PL']

print(f"Origin markets: {ORIGIN_COUNTRIES}")
print(f"Substitute destinations: {SUBSTITUTE_DESTINATIONS}")
print(f"Data directory: {os.path.abspath(DATA_DIR)}")

Origin markets: ['DE', 'UK', 'FR', 'NL', 'IT', 'ES', 'SE', 'AT', 'CZ']
Substitute destinations: ['CZ', 'HU', 'HR', 'EL']
Data directory: d:\Documentos\4º UNIVERSIDAD\TFG\Macro-Aviation-Tourism-Modeling\data\raw


---
## Phase 1 — Data Loading & Initial Exploration

We load each dataset, inspect its structure, and perform basic sanity checks. Eurostat CSVs follow a standard long format with columns like `freq`, `geo`, `TIME_PERIOD`, `OBS_VALUE`.

### 1.1 Target Variable — Overnight stays (`tour_occ_nim`)

In [10]:
# Load target variable
df_nights = pd.read_csv(PATHS['overnight_stays'])
print(f"Shape: {df_nights.shape}")
print(f"Columns: {list(df_nights.columns)}")
print(f"\nFirst rows:")
df_nights.head()

Shape: (9601, 11)
Columns: ['DATAFLOW', 'LAST UPDATE', 'freq', 'c_resid', 'unit', 'nace_r2', 'geo', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_FLAG', 'CONF_STATUS']

First rows:


,DATAFLOW,LAST UPDATE,freq,c_resid,unit,nace_r2,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:TOUR_OCC_NIM(1.0),02/02/26 23:00:00,Monthly,Domestic country,Number,Hotels and similar accommodation,European Union - 28 countries (2013-2020),2014-01,47790928.00,NaN,NaN
1,ESTAT:TOUR_OCC_NIM(1.0),02/02/26 23:00:00,Monthly,Domestic country,Number,Hotels and similar accommodation,European Union - 28 countries (2013-2020),2014-02,52885336.00,NaN,NaN
2,ESTAT:TOUR_OCC_NIM(1.0),02/02/26 23:00:00,Monthly,Domestic country,Number,Hotels and similar accommodation,European Union - 28 countries (2013-2020),2014-03,60211611.00,NaN,NaN
3,ESTAT:TOUR_OCC_NIM(1.0),02/02/26 23:00:00,Monthly,Domestic country,Number,Hotels and similar accommodation,European Union - 28 countries (2013-2020),2014-04,67177066.00,NaN,NaN
4,ESTAT:TOUR_OCC_NIM(1.0),02/02/26 23:00:00,Monthly,Domestic country,Number,Hotels and similar accommodation,European Union - 28 countries (2013-2020),2014-05,74346956.00,NaN,NaN


In [11]:
# Inspect unique values of key categorical columns
print("Unique 'geo':", df_nights['geo'].unique() if 'geo' in df_nights.columns else 'N/A')
print("\nUnique 'c_resid':", df_nights['c_resid'].nunique() if 'c_resid' in df_nights.columns else 'N/A')
print("\nUnique 'unit':", df_nights['unit'].unique() if 'unit' in df_nights.columns else 'N/A')
print("\nTime range:", df_nights['TIME_PERIOD'].min(), "to", df_nights['TIME_PERIOD'].max())
print("\nMissing OBS_VALUE:", df_nights['OBS_VALUE'].isna().sum())

Unique 'geo': ['European Union - 28 countries (2013-2020)' 'Poland']

Unique 'c_resid': 3

Unique 'unit': ['Number' 'Percentage change compared to same period in previous year'
 'Percentage change compared to same month in 2019'
 'Percentage change compared to same period two years ago']

Time range: 2003-01 to 2025-11

Missing OBS_VALUE: 0


In [7]:
# Filter: foreign tourists, absolute numbers (not % change), total accommodation
# Adjust filters based on your actual column values after inspecting above

# Example filter — ADAPT to your actual data values:
# df_target = df_nights[
#     (df_nights['unit'] == 'NR') &                    # absolute number of nights
#     (df_nights['c_resid'] != 'TOTAL') &              # not total (we want by country)
#     (df_nights['c_resid'] != 'DOM')                  # not domestic
# ].copy()

# For now, load and inspect — we'll refine the filter once we see the actual values
df_target = df_nights.copy()
print(f"Target shape after filtering: {df_target.shape}")

Target shape after filtering: (9601, 11)


### 1.2 Economic Variables

In [12]:
# --- HICP Monthly (primary CPI variable for relative price) ---
df_hicp = pd.read_csv(PATHS['hicp_monthly'])
print(f"HICP Monthly — Shape: {df_hicp.shape}")
print(f"Columns: {list(df_hicp.columns)}")
print(f"Countries: {sorted(df_hicp['geo'].unique()) if 'geo' in df_hicp.columns else 'N/A'}")
print(f"Time range: {df_hicp['TIME_PERIOD'].min()} to {df_hicp['TIME_PERIOD'].max()}")
df_hicp.head()

HICP Monthly — Shape: (4067, 10)
Columns: ['DATAFLOW', 'LAST UPDATE', 'freq', 'unit', 'coicop', 'geo', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_FLAG', 'CONF_STATUS']
Countries: ['Austria', 'Croatia', 'Czechia', 'European Union (EU6-1958, EU9-1973, EU10-1981, EU12-1986, EU15-1995, EU25-2004, EU27-2007, EU28-2013, EU27-2020)', 'France', 'Germany', 'Greece', 'Hungary', 'Italy', 'Netherlands', 'Poland', 'Spain', 'Sweden', 'United Kingdom', 'United States']
Time range: 2003-01 to 2025-12


,DATAFLOW,LAST UPDATE,freq,unit,coicop,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:PRC_HICP_MIDX(1.0),06/02/26 23:00:00,Monthly,"Index, 2015=100",All-items HICP,Austria,2003-01,78.71,NaN,NaN
1,ESTAT:PRC_HICP_MIDX(1.0),06/02/26 23:00:00,Monthly,"Index, 2015=100",All-items HICP,Austria,2003-02,78.85,NaN,NaN
2,ESTAT:PRC_HICP_MIDX(1.0),06/02/26 23:00:00,Monthly,"Index, 2015=100",All-items HICP,Austria,2003-03,79.06,NaN,NaN
3,ESTAT:PRC_HICP_MIDX(1.0),06/02/26 23:00:00,Monthly,"Index, 2015=100",All-items HICP,Austria,2003-04,78.98,NaN,NaN
4,ESTAT:PRC_HICP_MIDX(1.0),06/02/26 23:00:00,Monthly,"Index, 2015=100",All-items HICP,Austria,2003-05,78.88,NaN,NaN


In [13]:
# --- Exchange Rates (EUR bilateral) ---
df_exr = pd.read_csv(PATHS['exchange_rates'])
print(f"Exchange Rates — Shape: {df_exr.shape}")
print(f"Currencies: {df_exr['currency'].unique() if 'currency' in df_exr.columns else 'N/A'}")
print(f"Time range: {df_exr['TIME_PERIOD'].min()} to {df_exr['TIME_PERIOD'].max()}")
df_exr.head()

Exchange Rates — Shape: (1668, 10)
Currencies: ['Czech koruna' 'Pound sterling' 'Hungarian forint' 'Polish zloty'
 'Swedish krona' 'US dollar']
Time range: 2003-01 to 2026-02


,DATAFLOW,LAST UPDATE,freq,statinfo,unit,currency,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:ERT_BIL_EUR_M(1.0),03/03/26 11:00:00,Monthly,Average,National currency,Czech koruna,2003-01,31.49,NaN,NaN
1,ESTAT:ERT_BIL_EUR_M(1.0),03/03/26 11:00:00,Monthly,Average,National currency,Czech koruna,2003-02,31.64,NaN,NaN
2,ESTAT:ERT_BIL_EUR_M(1.0),03/03/26 11:00:00,Monthly,Average,National currency,Czech koruna,2003-03,31.75,NaN,NaN
3,ESTAT:ERT_BIL_EUR_M(1.0),03/03/26 11:00:00,Monthly,Average,National currency,Czech koruna,2003-04,31.62,NaN,NaN
4,ESTAT:ERT_BIL_EUR_M(1.0),03/03/26 11:00:00,Monthly,Average,National currency,Czech koruna,2003-05,31.39,NaN,NaN


In [14]:
# --- Real GDP (quarterly — will need interpolation to monthly) ---
df_gdp = pd.read_csv(PATHS['real_gdp'])
print(f"Real GDP — Shape: {df_gdp.shape}")
print(f"Countries: {sorted(df_gdp['geo'].unique()) if 'geo' in df_gdp.columns else 'N/A'}")
print(f"Time range: {df_gdp['TIME_PERIOD'].min()} to {df_gdp['TIME_PERIOD'].max()}")
df_gdp.head()

Real GDP — Shape: (1214, 11)
Countries: ['Austria', 'Croatia', 'Czechia', 'Germany', 'Hungary', 'Italy', 'Poland', 'Spain', 'Sweden', 'United Kingdom']
Time range: 1995-Q1 to 2025-Q4


,DATAFLOW,LAST UPDATE,freq,unit,s_adj,na_item,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:NAMQ_10_GDP(1.0),20/02/26 23:00:00,Quarterly,"Chain linked volumes (2010), million euro",Seasonally and calendar adjusted data,Gross domestic product at market prices,Austria,1995-Q1,53758.10,NaN,NaN
1,ESTAT:NAMQ_10_GDP(1.0),20/02/26 23:00:00,Quarterly,"Chain linked volumes (2010), million euro",Seasonally and calendar adjusted data,Gross domestic product at market prices,Austria,1995-Q2,54452.30,NaN,NaN
2,ESTAT:NAMQ_10_GDP(1.0),20/02/26 23:00:00,Quarterly,"Chain linked volumes (2010), million euro",Seasonally and calendar adjusted data,Gross domestic product at market prices,Austria,1995-Q3,54762.30,NaN,NaN
3,ESTAT:NAMQ_10_GDP(1.0),20/02/26 23:00:00,Quarterly,"Chain linked volumes (2010), million euro",Seasonally and calendar adjusted data,Gross domestic product at market prices,Austria,1995-Q4,55210.90,NaN,NaN
4,ESTAT:NAMQ_10_GDP(1.0),20/02/26 23:00:00,Quarterly,"Chain linked volumes (2010), million euro",Seasonally and calendar adjusted data,Gross domestic product at market prices,Austria,1996-Q1,55210.70,NaN,NaN


### 1.3 Transport Variables

In [15]:
# --- Aviation: Seats available (primary connectivity variable) ---
df_avia = pd.read_csv(PATHS['avia_seats'])
print(f"Aviation Seats — Shape: {df_avia.shape}")
print(f"Columns: {list(df_avia.columns)}")
# Inspect what measures are available (seats, flights, passengers)
if 'tra_meas' in df_avia.columns:
    print(f"Measures: {df_avia['tra_meas'].unique()}")
print(f"Time range: {df_avia['TIME_PERIOD'].min()} to {df_avia['TIME_PERIOD'].max()}")
df_avia.head()

Aviation Seats — Shape: (6574, 12)
Columns: ['DATAFLOW', 'LAST UPDATE', 'freq', 'unit', 'tra_meas', 'tra_cov', 'aircraft', 'rep_airp', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_FLAG', 'CONF_STATUS']
Measures: ['Commercial passenger air flights' 'Passengers on board'
 'Passengers seats available']
Time range: 2010-01 to 2025-10


,DATAFLOW,LAST UPDATE,freq,unit,tra_meas,tra_cov,aircraft,rep_airp,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:AVIA_TF_ACA(1.0),25/02/26 23:00:00,Monthly,Flight,Commercial passenger air flights,Total transport,Total,BYDGOSZCZ/SZWEDEROWO airport,2011-01,196,NaN,NaN
1,ESTAT:AVIA_TF_ACA(1.0),25/02/26 23:00:00,Monthly,Flight,Commercial passenger air flights,Total transport,Total,BYDGOSZCZ/SZWEDEROWO airport,2011-02,184,NaN,NaN
2,ESTAT:AVIA_TF_ACA(1.0),25/02/26 23:00:00,Monthly,Flight,Commercial passenger air flights,Total transport,Total,BYDGOSZCZ/SZWEDEROWO airport,2011-03,224,NaN,NaN
3,ESTAT:AVIA_TF_ACA(1.0),25/02/26 23:00:00,Monthly,Flight,Commercial passenger air flights,Total transport,Total,BYDGOSZCZ/SZWEDEROWO airport,2011-04,194,NaN,NaN
4,ESTAT:AVIA_TF_ACA(1.0),25/02/26 23:00:00,Monthly,Flight,Commercial passenger air flights,Total transport,Total,BYDGOSZCZ/SZWEDEROWO airport,2011-05,222,NaN,NaN


In [ ]:
# --- Aviation: Passengers by country pair ---
df_pax = pd.read_csv(PATHS['avia_passengers'])
print(f"Aviation Passengers — Shape: {df_pax.shape}")
print(f"Country pairs: {df_pax['geo'].unique() if 'geo' in df_pax.columns else 'N/A'}")
df_pax.head()

### 1.4 Demographic & Supplementary

In [ ]:
# --- Consumer Confidence Indicator ---
df_cci = pd.read_csv(PATHS['consumer_confidence'])
print(f"Consumer Confidence — Shape: {df_cci.shape}")
print(f"Countries: {sorted(df_cci['geo'].unique()) if 'geo' in df_cci.columns else 'N/A'}")
df_cci.head()

In [ ]:
# === SUMMARY TABLE ===
summary = pd.DataFrame({
    'Dataset': ['Overnight stays', 'HICP monthly', 'Exchange rates', 'Real GDP', 
                'Aviation seats', 'Aviation passengers', 'Consumer confidence'],
    'Frequency': ['Monthly', 'Monthly', 'Monthly', 'Quarterly', 
                  'Monthly', 'Monthly', 'Monthly'],
    'Rows': [len(df_nights), len(df_hicp), len(df_exr), len(df_gdp),
             len(df_avia), len(df_pax), len(df_cci)],
    'Period': [
        f"{df_nights['TIME_PERIOD'].min()} → {df_nights['TIME_PERIOD'].max()}",
        f"{df_hicp['TIME_PERIOD'].min()} → {df_hicp['TIME_PERIOD'].max()}",
        f"{df_exr['TIME_PERIOD'].min()} → {df_exr['TIME_PERIOD'].max()}",
        f"{df_gdp['TIME_PERIOD'].min()} → {df_gdp['TIME_PERIOD'].max()}",
        f"{df_avia['TIME_PERIOD'].min()} → {df_avia['TIME_PERIOD'].max()}",
        f"{df_pax['TIME_PERIOD'].min()} → {df_pax['TIME_PERIOD'].max()}",
        f"{df_cci['TIME_PERIOD'].min()} → {df_cci['TIME_PERIOD'].max()}",
    ]
})
summary

---
## Phase 2 — Descriptive Statistics & Univariate Analysis

### 2.1 Target variable: overnight stays in Poland by foreign tourists

We analyse the aggregate series first, then break it down by country of origin.

In [ ]:
# ---------------------------------------------------------------
# PREPARE THE TARGET SERIES
# ---------------------------------------------------------------
# TODO: Adapt these filters once you've inspected df_nights above.
# The goal is to get a clean monthly time series of TOTAL foreign 
# overnight stays in Poland (aggregated across all origin countries).

# Example (adapt column names/values):
# df_total = df_target[
#     (df_target['c_resid'] == 'FOR_TOTAL') &    # all foreign combined
#     (df_target['nace_r2'] == 'I551-I553') &     # all accommodation types
#     (df_target['unit'] == 'NR')
# ][['TIME_PERIOD', 'OBS_VALUE']].copy()

# Placeholder — replace with actual filter:
df_total = df_target.groupby('TIME_PERIOD')['OBS_VALUE'].sum().reset_index()
df_total['TIME_PERIOD'] = pd.to_datetime(df_total['TIME_PERIOD'])
df_total = df_total.sort_values('TIME_PERIOD').set_index('TIME_PERIOD')
df_total.columns = ['nights']

print(f"Target series: {len(df_total)} months")
print(f"Period: {df_total.index.min()} to {df_total.index.max()}")
df_total.describe()

In [ ]:
# --- TIME SERIES PLOT ---
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Raw series
axes[0].plot(df_total.index, df_total['nights'], color='#2c3e50', linewidth=1.2)
axes[0].set_title('Foreign overnight stays in Poland — Monthly', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Nights')
axes[0].axvspan('2020-03', '2021-06', alpha=0.15, color='red', label='COVID-19')
axes[0].legend(loc='upper left')

# Year-over-year growth
df_total['yoy'] = df_total['nights'].pct_change(12) * 100
axes[1].bar(df_total.index, df_total['yoy'], color=np.where(df_total['yoy'] >= 0, '#27ae60', '#e74c3c'), width=25)
axes[1].set_title('Year-over-Year Growth (%)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('% change')
axes[1].axhline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'target_series_overview.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- SEASONAL DECOMPOSITION ---
# Using multiplicative model (tourism has proportional seasonality)
decomposition = seasonal_decompose(df_total['nights'].dropna(), model='multiplicative', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomposition.observed.plot(ax=axes[0], color='#2c3e50')
axes[0].set_title('Observed', fontweight='bold')
decomposition.trend.plot(ax=axes[1], color='#e67e22')
axes[1].set_title('Trend', fontweight='bold')
decomposition.seasonal.plot(ax=axes[2], color='#2980b9')
axes[2].set_title('Seasonal', fontweight='bold')
decomposition.resid.plot(ax=axes[3], color='#95a5a6')
axes[3].set_title('Residual', fontweight='bold')

plt.suptitle('Seasonal Decomposition — Multiplicative Model', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'seasonal_decomposition.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- MONTHLY BOXPLOT (SEASONALITY PATTERN) ---
df_total['month'] = df_total.index.month
df_total['year'] = df_total.index.year

fig, ax = plt.subplots(figsize=(12, 5))
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
df_total.boxplot(column='nights', by='month', ax=ax, 
                 patch_artist=True,
                 boxprops=dict(facecolor='#3498db', alpha=0.5))
ax.set_xticklabels(month_names)
ax.set_title('Monthly distribution of overnight stays', fontsize=13, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Nights')
plt.suptitle('')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'monthly_boxplot.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- ACF / PACF ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df_total['nights'].dropna(), lags=36, ax=axes[0], title='ACF — Overnight stays')
plot_pacf(df_total['nights'].dropna(), lags=36, ax=axes[1], title='PACF — Overnight stays')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'acf_pacf_target.png'), dpi=150, bbox_inches='tight')
plt.show()

### 2.2 Breakdown by country of origin

Which origin markets are most important for Poland's inbound tourism?

In [ ]:
# --- TOP ORIGIN COUNTRIES ---
# TODO: adapt the filter to get overnight stays by c_resid (country of residence)
# Example:
# df_by_country = df_target[
#     (df_target['unit'] == 'NR') &
#     (df_target['c_resid'].isin(ORIGIN_COUNTRIES))
# ].groupby(['c_resid', 'TIME_PERIOD'])['OBS_VALUE'].sum().reset_index()

# Placeholder — replace once data is properly filtered:
# For now we create an example structure
print("TODO: Build country-level breakdown once df_target filters are confirmed.")
print("Expected output: stacked area chart of top origin markets over time.")

In [ ]:
# --- HEATMAP: MONTHLY NIGHTS BY ORIGIN COUNTRY (template) ---
# Once df_by_country is ready:
# pivot = df_by_country.pivot_table(index='month', columns='c_resid', values='OBS_VALUE', aggfunc='mean')
# fig, ax = plt.subplots(figsize=(12, 6))
# sns.heatmap(pivot, cmap='YlOrRd', annot=True, fmt='.0f', ax=ax)
# ax.set_title('Average monthly overnight stays by origin country')
# plt.tight_layout()
# plt.show()
print("Template ready — activate once country-level data is filtered.")

---
## Phase 3 — Multivariate EDA

Analyse relationships between the target variable and the candidate exogenous variables. Following Xie et al. (2021) and Salamanis et al. (2022), we compute:

1. **Pearson correlation** (contemporaneous) between each exogenous variable and overnight stays
2. **Cross-correlation with lag** — identify the optimal lag for each variable (à la Xie et al.)
3. **Correlation heatmap** across all variables

### 3.1 Build the monthly panel

We need to align all variables to the same monthly frequency and common time window.

In [ ]:
# ---------------------------------------------------------------
# HELPER: Standardise Eurostat TIME_PERIOD to datetime
# ---------------------------------------------------------------
def parse_eurostat_date(df, col='TIME_PERIOD'):
    """Convert Eurostat TIME_PERIOD (e.g. '2020-01', '2020Q1') to datetime."""
    df = df.copy()
    # Monthly format: '2020-01'
    try:
        df[col] = pd.to_datetime(df[col], format='%Y-%m')
    except:
        # Quarterly format: '2020-Q1' → convert to month start
        df[col] = pd.PeriodIndex(df[col], freq='Q').to_timestamp()
    return df

# ---------------------------------------------------------------
# HELPER: Quarterly to Monthly interpolation (cubic spline)
# ---------------------------------------------------------------
def quarterly_to_monthly(df, value_col='OBS_VALUE', date_col='TIME_PERIOD'):
    """Interpolate quarterly data to monthly using cubic spline.
    Following the approach noted in the data documentation (Song et al. framework)."""
    df = df.set_index(date_col).sort_index()
    monthly_idx = pd.date_range(df.index.min(), df.index.max(), freq='MS')
    df_monthly = df[value_col].reindex(monthly_idx).interpolate(method='cubic')
    return df_monthly.to_frame(name=value_col)

print("Helper functions defined.")

In [ ]:
# ---------------------------------------------------------------
# BUILD PANEL: This section will be completed once we confirm
# the exact column names and filter values from Phase 1.
# ---------------------------------------------------------------

# TEMPLATE for assembling the monthly master panel:
# 
# panel = pd.DataFrame(index=pd.date_range('2011-01', '2025-11', freq='MS'))
# 
# # Target
# panel['nights_total'] = df_total['nights']
# 
# # HICP by country (pivot from long to wide)
# hicp_wide = df_hicp_filtered.pivot_table(
#     index='TIME_PERIOD', columns='geo', values='OBS_VALUE'
# )
# hicp_wide.columns = [f'hicp_{c}' for c in hicp_wide.columns]
# panel = panel.join(hicp_wide)
# 
# # Exchange rates
# exr_wide = df_exr_filtered.pivot_table(
#     index='TIME_PERIOD', columns='currency', values='OBS_VALUE'
# )
# exr_wide.columns = [f'exr_{c}' for c in exr_wide.columns]
# panel = panel.join(exr_wide)
# 
# # Real GDP (interpolated from quarterly)
# for country in ORIGIN_COUNTRIES:
#     gdp_q = df_gdp[df_gdp['geo'] == country][['TIME_PERIOD', 'OBS_VALUE']]
#     gdp_m = quarterly_to_monthly(gdp_q)
#     panel[f'gdp_{country}'] = gdp_m['OBS_VALUE']
# 
# # Aviation seats
# panel['avia_seats_total'] = avia_seats_monthly
# 
# # Consumer confidence
# panel = panel.join(cci_wide)
# 
# panel.to_csv(os.path.join(PROCESSED_DIR, 'master_panel_monthly.csv'))

print("Panel assembly template ready — execute once Phase 1 filters are confirmed.")

### 3.2 Cross-correlation analysis with optimal lag

Following **Xie et al. (2021)**, we compute Pearson correlation between the target variable and each exogenous variable at lags 0 through 12 months. This reveals:
- How far in advance each variable "leads" tourism demand
- Which variables have the strongest predictive signal

In [ ]:
def cross_correlation_analysis(target_series, exog_series, max_lag=12, var_name='variable'):
    """
    Compute Pearson correlation between target and exogenous variable at lags 0..max_lag.
    
    Following Xie et al. (2021), Eq. 10: correlation between x_t (exog at time t) 
    and y_{t+l} (target at time t+l), for l = 0, 1, ..., L.
    
    A positive lag means the exogenous variable LEADS the target.
    
    Returns DataFrame with lag, correlation, and p-value.
    """
    results = []
    for lag in range(max_lag + 1):
        if lag == 0:
            x = exog_series
            y = target_series
        else:
            x = exog_series.iloc[:-lag]
            y = target_series.iloc[lag:]
        
        # Align indices
        common_idx = x.index.intersection(y.index)
        if len(common_idx) < 10:
            continue
        x_aligned = x.loc[common_idx].dropna()
        y_aligned = y.loc[common_idx].dropna()
        common = x_aligned.index.intersection(y_aligned.index)
        
        if len(common) >= 10:
            r, p = stats.pearsonr(x_aligned.loc[common], y_aligned.loc[common])
            results.append({'variable': var_name, 'lag': lag, 'correlation': r, 'p_value': p})
    
    return pd.DataFrame(results)

# Example usage (once panel is built):
# results = []
# for col in panel.columns:
#     if col != 'nights_total' and not panel[col].isna().all():
#         r = cross_correlation_analysis(panel['nights_total'], panel[col], var_name=col)
#         results.append(r)
# df_xcorr = pd.concat(results, ignore_index=True)
# 
# # Best lag per variable
# best_lags = df_xcorr.loc[df_xcorr.groupby('variable')['correlation'].idxmax()]
# best_lags.sort_values('correlation', ascending=False)

print("Cross-correlation function defined. Ready to use with panel data.")

In [ ]:
# --- VISUALIZATION: Cross-correlation heatmap (template) ---
# Once df_xcorr is computed:
# 
# pivot_xcorr = df_xcorr.pivot_table(index='variable', columns='lag', values='correlation')
# 
# fig, ax = plt.subplots(figsize=(16, 8))
# sns.heatmap(pivot_xcorr, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
#             ax=ax, vmin=-1, vmax=1, linewidths=0.5)
# ax.set_title('Cross-correlation: Exogenous variables vs. Overnight stays (by lag)')
# ax.set_xlabel('Lag (months)')
# plt.tight_layout()
# plt.savefig(os.path.join(FIGURES_DIR, 'cross_correlation_heatmap.png'), dpi=150)
# plt.show()

print("Heatmap template ready.")

---
## Phase 4 — Statistical Tests

### 4.1 Stationarity — Augmented Dickey-Fuller (ADF)

Time series models require stationary input. We test each variable and determine the order of differencing needed.

In [ ]:
def adf_test(series, name=''):
    """Run ADF test and return results as a dict."""
    result = adfuller(series.dropna(), autolag='AIC')
    return {
        'Variable': name,
        'ADF Statistic': result[0],
        'p-value': result[1],
        'Lags Used': result[2],
        'Stationary (5%)': 'Yes' if result[1] < 0.05 else 'No'
    }

# Once panel is built:
# adf_results = []
# for col in panel.columns:
#     adf_results.append(adf_test(panel[col], name=col))
# pd.DataFrame(adf_results)

# Test on target variable
adf_results = [adf_test(df_total['nights'], name='nights_total')]

# Also test first difference
df_total['nights_diff'] = df_total['nights'].diff()
adf_results.append(adf_test(df_total['nights_diff'].dropna(), name='nights_total (1st diff)'))

# And log + first difference
df_total['nights_log_diff'] = np.log(df_total['nights']).diff()
adf_results.append(adf_test(df_total['nights_log_diff'].dropna(), name='log(nights) (1st diff)'))

pd.DataFrame(adf_results)

### 4.2 Granger Causality Tests

Following **Xie et al. (2021)**: We test whether each exogenous variable Granger-causes overnight stays. This is a stronger test than simple correlation — it checks if past values of X improve the prediction of Y beyond what past values of Y alone can do.

This is a key step for **formal feature selection** (as recommended by the reviewed papers).

In [ ]:
def granger_test(target, exog, max_lag=6, var_name=''):
    """
    Run Granger causality test: does exog Granger-cause target?
    Returns the minimum p-value across tested lags and the best lag.
    """
    data = pd.DataFrame({'target': target, 'exog': exog}).dropna()
    if len(data) < max_lag * 3:
        return {'Variable': var_name, 'Best Lag': None, 'Min p-value': None, 'Granger-causes (5%)': 'Insufficient data'}
    
    try:
        result = grangercausalitytests(data[['target', 'exog']], maxlag=max_lag, verbose=False)
        p_values = {lag: result[lag][0]['ssr_ftest'][1] for lag in range(1, max_lag + 1)}
        best_lag = min(p_values, key=p_values.get)
        min_p = p_values[best_lag]
        return {
            'Variable': var_name,
            'Best Lag': best_lag,
            'Min p-value': round(min_p, 4),
            'Granger-causes (5%)': 'Yes' if min_p < 0.05 else 'No'
        }
    except Exception as e:
        return {'Variable': var_name, 'Best Lag': None, 'Min p-value': None, 'Granger-causes (5%)': str(e)}

# Once panel is built:
# granger_results = []
# for col in panel.columns:
#     if col != 'nights_total':
#         granger_results.append(granger_test(panel['nights_total'], panel[col], var_name=col))
# pd.DataFrame(granger_results).sort_values('Min p-value')

print("Granger causality function defined. Ready to use with panel data.")

### 4.3 Feature Selection Summary

Combining cross-correlation (Phase 3.2), Granger causality (Phase 4.2), and domain knowledge, we select the final set of exogenous variables and their optimal lags for the forecasting models.

| Variable | Best Lag | Pearson r | Granger p-value | Selected |
|----------|----------|-----------|-----------------|----------|
| *to be filled after analysis* | | | | |

This two-stage approach (correlation filtering + Granger validation) is inspired by **Wei et al. (2026)** and **Xie et al. (2021)**.

---
## Phase 5 — Feature Engineering

### 5.1 Relative Tourism Price (Song et al., 2010)

The key price competitiveness variable:

$$P_{it} = \frac{CPI_{PL}}{CPI_i} \times \frac{EX_i}{EX_{PL}}$$

where CPI is the All-items HICP and EX is the exchange rate vs. EUR.

For Eurozone origin countries (DE, FR, NL, IT, ES, AT, EL), the exchange rate ratio = 1, so the formula simplifies to just the CPI ratio.

In [ ]:
def compute_relative_price(hicp_pl, hicp_origin, exr_origin=None, exr_pl=None, eurozone=True):
    """
    Compute relative tourism price P_it = (CPI_PL / CPI_i) * (EX_i / EX_PL)
    
    For Eurozone origins: P_it = CPI_PL / CPI_i  (exchange rates cancel out)
    For non-Eurozone: full formula applies
    
    Parameters:
    -----------
    hicp_pl : Series — HICP index for Poland
    hicp_origin : Series — HICP index for origin country i
    exr_origin : Series — Exchange rate of origin currency per EUR (optional)
    exr_pl : Series — Exchange rate of PLN per EUR (optional)
    eurozone : bool — whether origin country is in Eurozone
    """
    cpi_ratio = hicp_pl / hicp_origin
    
    if eurozone or exr_origin is None:
        return cpi_ratio
    else:
        exr_ratio = exr_origin / exr_pl
        return cpi_ratio * exr_ratio

# Usage example (once data is loaded):
# for country in ORIGIN_COUNTRIES:
#     is_ez = country in EUROZONE
#     panel[f'relprice_{country}'] = compute_relative_price(
#         panel['hicp_PL'], panel[f'hicp_{country}'],
#         exr_origin=panel.get(f'exr_{country}'),
#         exr_pl=panel.get('exr_PLN'),
#         eurozone=is_ez
#     )

print("Relative price function defined.")

### 5.2 Additional engineered features

In [ ]:
# --- FEATURE ENGINEERING IDEAS ---
# (to be implemented once panel is ready)

# 1. Seasonal dummies (month indicators)
# panel['month'] = panel.index.month
# month_dummies = pd.get_dummies(panel['month'], prefix='month', drop_first=True)

# 2. Lag features of the target (autoregressive component)
# for lag in [1, 2, 3, 6, 12]:
#     panel[f'nights_lag{lag}'] = panel['nights_total'].shift(lag)

# 3. Rolling statistics
# panel['nights_roll3_mean'] = panel['nights_total'].rolling(3).mean()
# panel['nights_roll12_mean'] = panel['nights_total'].rolling(12).mean()

# 4. Year-over-year difference (removes seasonality)
# panel['nights_yoy_diff'] = panel['nights_total'].diff(12)

# 5. COVID dummy variable
# panel['covid'] = ((panel.index >= '2020-03') & (panel.index <= '2021-06')).astype(int)

# 6. Trend variable
# panel['trend'] = np.arange(len(panel))

print("Feature engineering templates ready.")

---
## Phase 6 — Baseline Models

Following the literature (Salamanis et al., 2022; Wei et al., 2026), we establish statistical baselines before moving to ML/DL:

1. **Naïve model** — forecast = same month last year
2. **Holt-Winters** (Triple Exponential Smoothing) — multiplicative seasonality
3. **SARIMA** — Seasonal ARIMA

### 6.1 Train/Test Split

We use the last 12–24 months as test set (out-of-sample evaluation). No future leakage.

In [ ]:
# --- TRAIN/TEST SPLIT ---
TEST_MONTHS = 12  # last 12 months for testing

# Once target series is confirmed:
train = df_total['nights'].iloc[:-TEST_MONTHS]
test = df_total['nights'].iloc[-TEST_MONTHS:]

print(f"Training: {train.index.min()} to {train.index.max()} ({len(train)} months)")
print(f"Testing:  {test.index.min()} to {test.index.max()} ({len(test)} months)")

In [ ]:
# --- EVALUATION METRICS ---
def evaluate_forecast(actual, predicted, model_name=''):
    """Compute RMSE, MAE, MAPE for a forecast."""
    actual = np.array(actual)
    predicted = np.array(predicted)
    mask = actual != 0  # avoid division by zero in MAPE
    
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    mape = np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100
    
    return {
        'Model': model_name,
        'RMSE': round(rmse, 2),
        'MAE': round(mae, 2),
        'MAPE (%)': round(mape, 2)
    }

results_table = []  # will accumulate results from all models
print("Evaluation function defined.")

### 6.2 Naïve model (seasonal)

In [ ]:
# Seasonal naïve: forecast = value from same month, 1 year ago
naive_forecast = train.iloc[-12:].values  # last 12 months of training

results_table.append(evaluate_forecast(test.values, naive_forecast, 'Naïve (seasonal)'))
print(results_table[-1])

### 6.3 Holt-Winters Exponential Smoothing

In [ ]:
# Holt-Winters with multiplicative seasonality (period=12)
try:
    hw_model = ExponentialSmoothing(
        train, 
        trend='add', 
        seasonal='mul', 
        seasonal_periods=12,
        use_boxcox=False
    ).fit(optimized=True)
    
    hw_forecast = hw_model.forecast(TEST_MONTHS)
    results_table.append(evaluate_forecast(test.values, hw_forecast.values, 'Holt-Winters'))
    print(results_table[-1])
    
    # Plot
    fig, ax = plt.subplots(figsize=(14, 5))
    train.plot(ax=ax, label='Train', color='#2c3e50')
    test.plot(ax=ax, label='Actual (test)', color='#2c3e50', linestyle='--')
    hw_forecast.plot(ax=ax, label='Holt-Winters forecast', color='#e74c3c')
    ax.set_title('Holt-Winters Forecast', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'hw_forecast.png'), dpi=150)
    plt.show()
except Exception as e:
    print(f"Holt-Winters error: {e}")
    print("Ensure the target series has enough data and no missing values.")

### 6.4 SARIMA

In [ ]:
# SARIMA(p,d,q)(P,D,Q,12)
# Starting with a reasonable configuration — can be optimized with AIC grid search

try:
    sarima_model = SARIMAX(
        train,
        order=(1, 1, 1),
        seasonal_order=(1, 1, 1, 12),
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit(disp=False)
    
    print(sarima_model.summary().tables[0])
    print(f"\nAIC: {sarima_model.aic:.2f}")
    print(f"BIC: {sarima_model.bic:.2f}")
    
    sarima_forecast = sarima_model.forecast(TEST_MONTHS)
    results_table.append(evaluate_forecast(test.values, sarima_forecast.values, 'SARIMA(1,1,1)(1,1,1,12)'))
    print(results_table[-1])
    
    # Plot
    fig, ax = plt.subplots(figsize=(14, 5))
    train[-36:].plot(ax=ax, label='Train (last 3y)', color='#2c3e50')
    test.plot(ax=ax, label='Actual (test)', color='#2c3e50', linestyle='--')
    sarima_forecast.plot(ax=ax, label='SARIMA forecast', color='#8e44ad')
    ax.fill_between(sarima_forecast.index,
                    sarima_model.get_forecast(TEST_MONTHS).conf_int().iloc[:, 0],
                    sarima_model.get_forecast(TEST_MONTHS).conf_int().iloc[:, 1],
                    alpha=0.2, color='#8e44ad')
    ax.set_title('SARIMA Forecast with 95% CI', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'sarima_forecast.png'), dpi=150)
    plt.show()
except Exception as e:
    print(f"SARIMA error: {e}")

---
## Phase 7 — Machine Learning Models

We now move to models that can incorporate exogenous variables. Following the progressive complexity approach recommended by the literature:

1. **Random Forest** — robust, handles non-linearity, good baseline for ML
2. **Gradient Boosting** (XGBoost / sklearn) — usually top performer for tabular data

### Important: Time Series Cross-Validation

We use `TimeSeriesSplit` to avoid future leakage. The test set remains untouched until final evaluation.

In [ ]:
# ---------------------------------------------------------------
# PREPARE FEATURES FOR ML
# ---------------------------------------------------------------
# This requires the master panel from Phase 5.
# Template below — activate once panel is ready.

def prepare_ml_features(panel, target_col='nights_total', test_months=12):
    """
    Prepare features and target for ML models.
    Includes: lags of target, exogenous variables, seasonal dummies.
    
    Returns: X_train, y_train, X_test, y_test
    """
    df = panel.copy()
    
    # Drop rows with NaN (from lagging)
    df = df.dropna()
    
    # Split
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    X_train, X_test = X.iloc[:-test_months], X.iloc[-test_months:]
    y_train, y_test = y.iloc[:-test_months], y.iloc[-test_months:]
    
    return X_train, y_train, X_test, y_test

print("ML preparation function defined.")

### 7.1 Random Forest

In [ ]:
# Once features are prepared:
# X_train, y_train, X_test, y_test = prepare_ml_features(panel)

# --- RANDOM FOREST ---
# rf_model = RandomForestRegressor(
#     n_estimators=500,
#     max_depth=10,
#     min_samples_leaf=5,
#     random_state=42,
#     n_jobs=-1
# )
# rf_model.fit(X_train, y_train)
# rf_pred = rf_model.predict(X_test)
# results_table.append(evaluate_forecast(y_test.values, rf_pred, 'Random Forest'))
# print(results_table[-1])

# --- FEATURE IMPORTANCE ---
# importance = pd.Series(rf_model.feature_importances_, index=X_train.columns)
# importance = importance.sort_values(ascending=True)
# 
# fig, ax = plt.subplots(figsize=(10, 8))
# importance.tail(20).plot(kind='barh', ax=ax, color='#2ecc71')
# ax.set_title('Random Forest — Top 20 Feature Importances', fontweight='bold')
# plt.tight_layout()
# plt.savefig(os.path.join(FIGURES_DIR, 'rf_feature_importance.png'), dpi=150)
# plt.show()

print("Random Forest template ready — activate once panel features are prepared.")

### 7.2 Gradient Boosting

In [ ]:
# --- GRADIENT BOOSTING ---
# gb_model = GradientBoostingRegressor(
#     n_estimators=500,
#     max_depth=5,
#     learning_rate=0.05,
#     subsample=0.8,
#     min_samples_leaf=5,
#     random_state=42
# )
# gb_model.fit(X_train, y_train)
# gb_pred = gb_model.predict(X_test)
# results_table.append(evaluate_forecast(y_test.values, gb_pred, 'Gradient Boosting'))
# print(results_table[-1])

print("Gradient Boosting template ready.")

---
## Phase 8 — Deep Learning: LSTM with Exogenous Variables

Following **Salamanis et al. (2022)**, we implement two LSTM variants:

1. **LSTM-Base** — only historical overnight stays (autoregressive)
2. **LSTM-Exog** — overnight stays + selected exogenous variables

This allows us to measure the **marginal value of exogenous data** for deep learning forecasting.

### Architecture decisions (from literature review):
- **Activation:** tanh (Salamanis et al., all LSTM layers)
- **Optimizer:** Adam (Wei et al.) or RMSProp (Salamanis et al.)
- **Strategy:** Direct multi-step forecasting (avoids error accumulation, per Salamanis et al.)
- **Hyperparameter tuning:** Grid search on validation set
- **Test:** Diebold-Mariano test for statistical significance (Wei et al.)

In [ ]:
# ---------------------------------------------------------------
# LSTM implementation will go here.
# Dependencies: tensorflow/keras or pytorch
# ---------------------------------------------------------------
# 
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
# from tensorflow.keras.callbacks import EarlyStopping
# from tensorflow.keras.optimizers import Adam
#
# def create_lstm_sequences(data, lookback=12, horizon=1):
#     """Create supervised learning sequences from time series.
#     Following Salamanis et al. (2022), Section 3.3."""
#     X, y = [], []
#     for i in range(lookback, len(data) - horizon + 1):
#         X.append(data[i - lookback:i])
#         y.append(data[i + horizon - 1])  # direct strategy
#     return np.array(X), np.array(y)
#
# def build_lstm_model(input_shape, units=64, layers=2, dropout=0.2):
#     model = Sequential()
#     model.add(LSTM(units, return_sequences=(layers > 1), 
#                    input_shape=input_shape, activation='tanh'))
#     if layers > 1:
#         model.add(Dropout(dropout))
#         model.add(LSTM(units // 2, activation='tanh'))
#     model.add(Dropout(dropout))
#     model.add(Dense(1, activation='linear'))
#     model.compile(optimizer=Adam(learning_rate=1e-3), loss='mse')
#     return model

print("LSTM templates defined. Implementation pending — requires TensorFlow/PyTorch.")
print("This will be the core modelling phase of the thesis.")

---
## Phase 9 — Model Comparison & Evaluation

### 9.1 Results Summary

In [ ]:
# --- RESULTS TABLE ---
df_results = pd.DataFrame(results_table)
print("\n=== FORECASTING PERFORMANCE COMPARISON ===\n")
print(df_results.to_string(index=False))

# Highlight best model per metric
if len(df_results) > 0:
    for metric in ['RMSE', 'MAE', 'MAPE (%)']:
        if metric in df_results.columns:
            best = df_results.loc[df_results[metric].idxmin()]
            print(f"\nBest {metric}: {best['Model']} ({best[metric]})")

In [ ]:
# --- VISUAL COMPARISON (template) ---
# Once all models have predictions:
# 
# fig, ax = plt.subplots(figsize=(14, 6))
# test.plot(ax=ax, label='Actual', color='#2c3e50', linewidth=2)
# 
# colors = {'Naïve (seasonal)': '#95a5a6', 'Holt-Winters': '#e74c3c', 
#           'SARIMA': '#8e44ad', 'Random Forest': '#27ae60',
#           'Gradient Boosting': '#f39c12', 'LSTM-Base': '#2980b9',
#           'LSTM-Exog': '#c0392b'}
# 
# for model_name, predictions in all_predictions.items():
#     ax.plot(test.index, predictions, label=model_name, 
#             color=colors.get(model_name, 'gray'), linewidth=1.2)
# 
# ax.set_title('All Models — Forecast Comparison', fontsize=14, fontweight='bold')
# ax.legend(loc='upper left')
# ax.set_ylabel('Overnight stays')
# plt.tight_layout()
# plt.savefig(os.path.join(FIGURES_DIR, 'model_comparison.png'), dpi=150)
# plt.show()

print("Comparison template ready.")

### 9.2 Diebold-Mariano Test

Statistical test to verify whether differences in forecast accuracy are significant (Wei et al., 2026).

In [ ]:
def diebold_mariano_test(actual, pred1, pred2, h=1, alternative='two-sided'):
    """
    Diebold-Mariano test for comparing forecast accuracy.
    H0: Both forecasts have equal accuracy.
    
    Parameters:
    -----------
    actual : array — actual values
    pred1, pred2 : arrays — predictions from two models
    h : int — forecast horizon (for HAC variance adjustment)
    alternative : str — 'two-sided', 'less', or 'greater'
    
    Returns: DM statistic, p-value
    """
    e1 = np.array(actual) - np.array(pred1)
    e2 = np.array(actual) - np.array(pred2)
    d = e1**2 - e2**2  # loss differential (MSE-based)
    
    n = len(d)
    d_mean = np.mean(d)
    
    # HAC variance (Newey-West with h-1 lags)
    gamma_0 = np.var(d, ddof=1)
    gamma_sum = 0
    for k in range(1, h):
        gamma_k = np.cov(d[k:], d[:-k])[0, 1] if len(d[k:]) > 1 else 0
        gamma_sum += gamma_k
    
    var_d = (gamma_0 + 2 * gamma_sum) / n
    
    if var_d <= 0:
        return np.nan, np.nan
    
    dm_stat = d_mean / np.sqrt(var_d)
    
    if alternative == 'two-sided':
        p_value = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    elif alternative == 'less':
        p_value = stats.norm.cdf(dm_stat)
    else:
        p_value = 1 - stats.norm.cdf(dm_stat)
    
    return round(dm_stat, 4), round(p_value, 4)

print("Diebold-Mariano test function defined.")

---
## Next Steps

After completing this pipeline:

1. **Thesis Chapter 4 (Descriptive Analysis):** Phases 2-3 generate the figures and tables
2. **Thesis Chapter 5 (Methodology):** Phases 4-5 document feature selection and engineering  
3. **Thesis Chapter 6 (Results):** Phases 6-9 provide all model results and comparisons
4. **Scenarios:** Once the best model is identified, run demand scenarios (e.g., exchange rate shock, capacity reduction, COVID-like event)

### Implementation priority:
1. ✅ Get Phase 1 working (load all data correctly)
2. ✅ Phase 2 — univariate analysis of target
3. ✅ Build the master panel (Phase 3.1 + 5)
4. Run cross-correlations and Granger tests (Phases 3.2 + 4)
5. Baselines (Phase 6)
6. ML models (Phase 7)
7. LSTM (Phase 8)
8. Final comparison (Phase 9)